In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras import regularizers
from tensorflow.keras import Model, layers, initializers
from tensorflow.keras import Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Dropout,Lambda
from tensorflow.keras.layers import BatchNormalization, Activation, ZeroPadding2D
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import UpSampling2D, Conv2D, Layer
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import SimpleRNN, Embedding
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import pandas as pd
import random
import time
import math
from math import ceil
import sys
import os
from sklearn import svm, datasets
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import label_binarize
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import scipy.stats as st
from itertools import cycle
import matplotlib.pyplot as plt
from matplotlib.pyplot import MultipleLocator
%matplotlib inline
from tensorflow.python.profiler.model_analyzer import profile
from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

In [ ]:
def split(data_set):
    feature = data_set.iloc[:,:-1]
    target = data_set.iloc[:,-1]
    return feature.values,target.values

def c_pairs(x, digit_indices, num_classes):
    pairs = []
    labels = []
    for d in range(num_classes):
        n = digit_indices[d].size-1
        for i in range(n):
            z1, z2 = digit_indices[d][i], digit_indices[d][i + 1]
            pairs += [[x[z1], x[z2]]]
            inc = random.randrange(1, num_classes)
            dn = (d + inc) % num_classes
            m = digit_indices[dn].size-1
            im = random.randint(0, m)
            z1, z2 = digit_indices[d][i], digit_indices[dn][im]
            pairs += [[x[z1], x[z2]]]
            labels += [1, 0]
    return np.array(pairs), np.array(labels)

def cp_onset(images, labels):
    num_classes = len(Counter(labels).keys())
    digit_indices = [np.where(labels == i)[0] for i in range(num_classes)]
    pairs, y = c_pairs(images, digit_indices, num_classes)
    y = y.astype('float32')
    return pairs, y

def initialize(input_dim):
    input_ = Input(shape=(input_dim,), name="base_input")
    x = Dense(5, activation='tanh', name="first_base_dense")(input_)
    outputs = Dense(64, activation='tanh', name="second_base_dense")(x)
    return Model(inputs=input_, outputs=outputs)

def euclidean_distance(vects):
    x, y = vects
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

euclidean_dis_layer_lambda = Lambda(euclidean_distance, name="outpur_layer")

def contrastive_loss_with_margin(margin):
    def contrastive_loss(y_true, y_pred):
        square_pred = K.square(y_pred)
        margin_square = K.square(K.maximum(margin - y_pred, 0))
        return K.mean(y_true * square_pred + (1 - y_true) * margin_square)
    return contrastive_loss

def comp_ac(y_true, y_pred):
    pred = y_pred.ravel() < 0.5
    return np.mean(pred == y_true)

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def min_max_norm(df, name):
    x = df[name].values.reshape(-1, 1)
    min_max_scaler = MinMaxScaler()
    x_scaled = min_max_scaler.fit_transform(x)
    df[name] = x_scaled

# def min_max_norm(df, name):
#   x = df[name].values.reshape(-1, 1)
#   min_max_scaler = MinMaxScaler()
#   x_scaled = min_max_scaler.fit_transform(x)
#   df[name] = x_scaled

# base_csv_path = r'sample_data/data_edge.csv'
# df = pd.read_csv(base_csv_path)
# df.drop('src_ip', axis=1, inplace=True)
# df.drop('dst_ip', axis=1, inplace=True)
# df.drop('src_port', axis=1, inplace=True)
# df.drop('dst_port', axis=1, inplace=True)
# df.drop('timestamp', axis=1, inplace=True)

base_path = r'sample_data/med/'
df = pd.DataFrame()
for csv_path in os.listdir(base_path):
  if csv_path.endswith('.csv'):
    tmp = pd.read_csv(os.path.join(base_path, csv_path))
    df = pd.concat([df, tmp], axis=0)

norm_cols = df.columns.values[:-2]
print(norm_cols)
for feature_id in norm_cols:
    min_max_norm(df, feature_id)

feature, target = split(df)
input_dim = feature.shape[1]
print(f"inputs dim is {input_dim}")
feature, feature_test, target, target_test = train_test_split(feature, target, test_size=0.25,stratify = target, random_state = 1)
print(feature.shape, target.shape, type(feature), type(target))
tr_pairs, tr_y = cp_onset(feature, target)
ts_pairs, ts_y = cp_onset(feature_test, target_test)

In [ ]:
base_network = initialize(input_dim)
print(base_network.summary())

forward_pass = tf.function(
    base_network.call,
    input_signature=[tf.TensorSpec(shape=(1,) + base_network.input_shape[1:])])

input_1 = Input(shape=(input_dim,), name="left_input")
embedding1 = base_network(input_1)
input_2 = Input(shape=(input_dim,), name="right_input")
embedding2 = base_network(input_2)
output = euclidean_dis_layer_lambda([embedding1, embedding2])
model = Model([input_1, input_2], output)

cosine_decay = tf.keras.optimizers.schedules.CosineDecay(
                initial_learning_rate=0.01, decay_steps=100)
exponential_decay = tf.keras.optimizers.schedules.ExponentialDecay(
                        initial_learning_rate=0.01, decay_steps=1000, decay_rate=0.96)
model.compile(loss=contrastive_loss_with_margin(margin=1)
              ,optimizer = tf.keras.optimizers.Adam(learning_rate=0.001))

history = model.fit([tr_pairs[:,0], tr_pairs[:,1]], tr_y, epochs=100, batch_size=1024, validation_data=([ts_pairs[:,0], ts_pairs[:,1]], ts_y))
# model.load_weights(r'.\weights\embedding_TonIoT-NI_Siamese_multi_gpu.h5')

loss = model.evaluate(x=[ts_pairs[:,0],ts_pairs[:,1]], y=ts_y)

y_pred_train = model.predict([tr_pairs[:,0], tr_pairs[:,1]])
train_accuracy = comp_ac(tr_y, y_pred_train)

y_pred_test = model.predict([ts_pairs[:,0], ts_pairs[:,1]])
test_accuracy = comp_ac(ts_y, y_pred_test)

print("Loss = {}, Train Accuracy = {} Test Accuracy = {}".format(loss, train_accuracy, test_accuracy))

In [ ]:
class PatchEmbed(layers.Layer):
    def __init__(self, img_size, patch_size, embed_dim):
        super(PatchEmbed, self).__init__()
        self.embed_dim = embed_dim
        self.img_size = (img_size, img_size)
        self.grid_size = (ceil(img_size / patch_size), ceil(img_size / patch_size))
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = layers.Conv2D(filters=embed_dim, kernel_size=patch_size,
                                  strides=patch_size, padding='SAME',
                                  kernel_initializer=initializers.LecunNormal(),
                                  bias_initializer=initializers.Zeros())

    def call(self, inputs, **kwargs):
        B = tf.shape(inputs)[0]
        H = self.img_size[0]
        W = self.img_size[1]
        C = tf.shape(inputs)[3]  # 1
        assert int(H) == self.img_size[0] and int(W) == self.img_size[1]
        x = self.proj(inputs)
        x = tf.reshape(x, [-1, self.num_patches, self.embed_dim])
        return x

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.P = np.zeros((1, max_len, num_hiddens))
        X = np.arange(max_len, dtype=np.float32).reshape(
            -1,1)/np.power(10000, np.arange(
            0, num_hiddens, 2, dtype=np.float32) / num_hiddens)
        self.P[:, :, 0::2] = np.sin(X)
        self.P[:, :, 1::2] = np.cos(X)

    def call(self, inputs, **kwargs):
        inputs = inputs + self.P[:, :inputs.shape[1], :]
        return self.dropout(inputs, **kwargs)

class Pooling(layers.Layer):
    def __init__(self, pool_size=2):
        super().__init__()
        self.pool = AveragePooling2D(
            pool_size=pool_size, strides=1, padding='same', name='pool')

    def call(self, inputs, training=None):
        B = tf.shape(inputs)[0]
        x = tf.reshape(inputs, [B, 4, 4, 8])
        x = self.pool(x)
        x = tf.reshape(x, [B, 16, 8])
        return x

class Attention(layers.Layer):
    k_ini = initializers.GlorotUniform()
    b_ini = initializers.Zeros()

    def __init__(self,
                 dim,
                 num_heads,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop_ratio=0.,
                 proj_drop_ratio=0.,
                 name='MultiHeadAttention'):
        super(Attention, self).__init__(name=name)
        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** (-0.5)
        self.qkv = layers.Dense(dim * 3, use_bias=qkv_bias, name="qkv",
                                kernel_initializer=self.k_ini, bias_initializer=self.b_ini)
        self.attn_drop = layers.Dropout(attn_drop_ratio)
        self.proj = layers.Dense(dim, name="out",
                                 kernel_initializer=self.k_ini, bias_initializer=self.b_ini)
        self.proj_drop = layers.Dropout(proj_drop_ratio)

    def call(self, inputs, training=None):
        B = tf.shape(inputs)[0]
        N = tf.shape(inputs)[1]
        C = self.dim
        qkv = self.qkv(inputs)
        qkv = tf.cast(qkv,'float32')
        qkv = tf.reshape(qkv, [B, N, 3, self.num_heads, C // self.num_heads])
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = tf.matmul(a=q, b=k, transpose_b=True) * self.scale
        attn = tf.nn.softmax(attn, axis=-1)
        attn = self.attn_drop(attn, training=training)
        x = tf.matmul(attn, v)
        x = tf.transpose(x, [0, 2, 1, 3])
        x = tf.reshape(x, [B, N, C])
        x = self.proj(x)
        x = self.proj_drop(x, training=training)
        return x

class MLP(layers.Layer):
    k_ini = initializers.GlorotUniform()
    b_ini = initializers.RandomNormal(stddev=1e-6)

    def __init__(self, in_features=8, mlp_ratio=8, drop=0., name=None):
        super(MLP, self).__init__(name=name)
        self.fc1 = layers.Dense(in_features * mlp_ratio, name="Dense_0",
                                kernel_initializer=self.k_ini, bias_initializer=self.b_ini)
        self.act = layers.Activation("relu")
        self.fc2 = layers.Dense(in_features, name="Dense_1",
                                kernel_initializer=self.k_ini, bias_initializer=self.b_ini)
        self.drop = layers.Dropout(drop)

    def call(self, inputs, training=None):
        x = self.fc1(inputs)
        x = self.act(x)
        x = self.drop(x, training=training)
        x = self.fc2(x)
        x = self.drop(x, training=training)
        return x

class MLP_p(layers.Layer):
    k_ini = initializers.GlorotUniform()
    b_ini = initializers.RandomNormal(stddev=1e-6)

    def __init__(self, in_features=8, mlp_ratio=0.5, drop=0., name=None):
        super(MLP_p, self).__init__(name=name)
        self.fc1 = layers.Dense(2, name="Dense_0",
                                kernel_initializer=self.k_ini, bias_initializer=self.b_ini)
        self.act = layers.Activation("relu")
        self.fc2 = layers.Dense(in_features, name="Dense_1",
                                kernel_initializer=self.k_ini, bias_initializer=self.b_ini)
        self.drop = layers.Dropout(drop)

    def call(self, inputs, training=None):
        x = self.fc1(inputs)
        x = self.act(x)
        x = self.drop(x, training=training)
        x = self.fc2(x)
        x = self.drop(x, training=training)
        return x

class Block(layers.Layer):
    def __init__(self,
                 dim,
                 num_heads,
                 qkv_bias=False,
                 qk_scale=None,
                 drop_ratio=0.,
                 attn_drop_ratio=0.,
                 drop_path_ratio=0.,
                 name=None):
        super(Block, self).__init__(name=name)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_0")
        self.attn = Attention(dim=dim, num_heads=num_heads,
                              qkv_bias=qkv_bias, qk_scale=qk_scale,
                              attn_drop_ratio=attn_drop_ratio, proj_drop_ratio=drop_ratio,
                              name="MultiHeadAttention")
        self.pool = Pooling()

        self.drop_path = layers.Dropout(rate=drop_path_ratio, noise_shape=(None, 1, 1)) if drop_path_ratio > 0. \
            else layers.Activation("linear")
        self.norm2 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_1")
        self.mlp = MLP(in_features=dim, drop=drop_ratio, name="MlpBlock")

    def call(self, inputs, training=None):
        x = inputs + self.drop_path(self.attn(self.norm1(inputs)), training=training)
        x = x + self.drop_path(self.mlp(self.norm2(x)), training=training)
        return x

class Block_p(layers.Layer):
    def __init__(self,
                 dim,
                 num_heads,
                 qkv_bias=False,
                 qk_scale=None,
                 drop_ratio=0.1,
                 attn_drop_ratio=0.,
                 drop_path_ratio=0.,
                 name=None):
        super(Block_p, self).__init__(name=name)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_0")
        self.attn = Attention(dim=dim, num_heads=num_heads,
                              qkv_bias=qkv_bias, qk_scale=qk_scale,
                              attn_drop_ratio=attn_drop_ratio, proj_drop_ratio=drop_ratio,
                              name="MultiHeadAttention")
        self.pool = Pooling()
        self.drop_path = layers.Dropout(rate=drop_path_ratio, noise_shape=(None, 1, 1)) if drop_path_ratio > 0. \
            else layers.Activation("linear")
        self.norm2 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_1")
        self.mlp = MLP_p(in_features=dim, drop=drop_ratio, name="MlpBlock")

    def call(self, inputs, training=None):
        x = inputs + self.drop_path(self.pool(self.norm1(inputs)), training=training)
        x = x + self.drop_path(self.mlp(self.norm2(x)), training=training)
        return x

class Choice1(Layer):
    def __init__(self, **kwargs):
        super(Choice1, self).__init__(**kwargs)
        self.supports_masking = True

    def compute_mask(self, inputs, mask=None):
        if mask is not None:
            return mask[1]

    def call(self, inputs):
        source, target = inputs
        mask = K.random_bernoulli(shape=[1], p=0.1)
        output = mask * source + (1 - mask) * target
        return K.in_train_phase(output, target)

    def compute_output_shape(self, input_shape):
        return input_shape[1]

class Choice2(Layer):
    def __init__(self, base_replacing_rate=0, k=0.005,  **kwargs):
        super(BinaryRandomChoice2, self).__init__(**kwargs)
        self.supports_masking = True
        self.b = base_replacing_rate
        self.k = k
        self.step_counter = 0

    def call(self, inputs):
        pre, succ = inputs
        self.step_counter += 1
        p = min(self.k * self.step_counter + self.b, 1.0)
        mask = K.random_bernoulli(shape=[1], p=p)
        output = (1 - mask) * pre + mask * succ
        return K.in_train_phase(output, succ)

class Choice3(Layer):
    def __init__(self, proportion=0.5, **kwargs):
        super(BinaryRandomChoice3, self).__init__(**kwargs)
        self.proportion = proportion

    def call(self, inputs):
        pre, succ = inputs
        pre = pre * self.proportion
        succ = succ * (1 - self.proportion)
        output = (pre + succ)
        return output

In [ ]:
class Theseus(Model):
    def __init__(self, InputShape =(8,8,1),
                 img_size=8, patch_size=2,
                 embed_dim=8, num_heads=2,
                 pre_num_blocks=12, succ_num_blocks=3,
                 qkv_bias=True, qk_scale=None,
                 drop_ratio=0., drop_ratio_p=0.,
                 attn_drop_ratio=0., drop_path_ratio=0.,
                 num_classes=4,
                 name="Theseus"):
        super(Theseus, self).__init__(name=name)
        self.InputShape = InputShape
        self.img_size = img_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.drop_ratio = drop_ratio
        self.drop_ratio_p = drop_ratio_p
        self.attn_drop_ratio = attn_drop_ratio
        self.drop_path_ratio = drop_path_ratio
        self.num_classes = num_classes

        self.p_dpr = np.linspace(0., drop_path_ratio, pre_num_blocks)
        self.s_dpr = np.linspace(0., drop_path_ratio, succ_num_blocks)

        self.Pre_Embed = self.Build_Pre_Embed()
        self.Succ_Embed = self.Build_Succ_Embed()

        self.Pre_blocks1 = self.Build_Pre1()
        self.Pre_blocks2 = self.Build_Pre2()
        self.Pre_blocks3 = self.Build_Pre3()

        self.Succ_blocks1 = self.Build_Succ1()
        self.Succ_blocks2 = self.Build_Succ2()
        self.Succ_blocks3 = self.Build_Succ3()

        self.Pre_Classify = self.Build_Pre_Classify()
        self.Succ_Classify = self.Build_Succ_Classify()

        self.Predecessor = Sequential()
        self.Predecessor.add(self.Pre_Embed)
        self.Predecessor.add(self.Pre_blocks1)
        self.Predecessor.add(self.Pre_blocks2)
        self.Predecessor.add(self.Pre_blocks3)
        self.Predecessor.add(self.Pre_Classify)

        self.Successor = Sequential()
        self.Successor.add(self.Succ_Embed)
        self.Successor.add(self.Succ_blocks1)
        self.Successor.add(self.Succ_blocks2)
        self.Successor.add(self.Succ_blocks3)
        self.Successor.add(self.Succ_Classify)

    def Build_Pre_Embed(self):
        inputs = Input(shape=self.InputShape)
        x = PatchEmbed(img_size=self.img_size, patch_size=self.patch_size, embed_dim=self.embed_dim)(inputs) # ->[B,16,8]
        x = PositionalEncoding(num_hiddens=self.embed_dim, dropout=0.1)(x)
        outputs = layers.Dropout(self.drop_ratio)(x)
        print(f"Pre_Embed out shape is {outputs.shape}")
        model = Model(inputs, outputs)
        return model

    def Build_Succ_Embed(self):
        inputs = Input(shape=self.InputShape)
        x = PatchEmbed(img_size=self.img_size, patch_size=self.patch_size, embed_dim=self.embed_dim)(inputs) # ->[B,16,8]
        x = PositionalEncoding(num_hiddens=self.embed_dim, dropout=0.1)(x)
        outputs = layers.Dropout(self.drop_ratio)(x)
        model = Model(inputs, outputs)
        return model

    def Build_Pre1(self):
        inputs = Input(shape=(16,8))
        x = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[0], name="pre_block1")(inputs)
        x = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[1], name="pre_block2")(x)
        outputs = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[2], name="pre_block3")(x)
        print(f"Pre1 out shape is {outputs.shape}")
        model = Model(inputs, outputs)
        return model

    def Build_Pre2(self):
        inputs = Input(shape=(16,8))
        x = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[3], name="pre_block4")(inputs)
        x = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[4], name="pre_block5")(x)
        outputs = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[5], name="pre_block6")(x)
        print(f"Pre2 out shape is {outputs.shape}")
        model = Model(inputs, outputs)
        return model

    def Build_Pre3(self):
        inputs = Input(shape=(16,8))
        x = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[6], name="pre_block7")(inputs)
        x = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[7], name="pre_block8")(x)
        outputs = Block(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.p_dpr[8], name="pre_block9")(x)
        print(f"Pre3 out shape is {outputs.shape}")
        model = Model(inputs, outputs)
        return model

    def Build_Succ1(self):
        inputs = Input(shape=(16,8))
        outputs = Block_p(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio_p, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.s_dpr[0], name="succ_block1")(inputs)
        model = Model(inputs, outputs)
        return model

    def Build_Succ2(self):
        inputs = Input(shape=(16,8))
        outputs = Block_p(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio_p, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.s_dpr[1], name="succ_block2")(inputs)
        model = Model(inputs, outputs)
        return model

    def Build_Succ3(self):
        inputs = Input(shape=(16,8))
        outputs = Block_p(dim=self.embed_dim, num_heads=self.num_heads, qkv_bias=self.qkv_bias,
                             qk_scale=self.qk_scale, drop_ratio=self.drop_ratio_p, attn_drop_ratio=self.attn_drop_ratio,
                             drop_path_ratio=self.s_dpr[2], name="succ_block3")(inputs)
        model = Model(inputs, outputs)
        return model

    def Build_Pre_Classify(self):
        inputs = Input(shape=(16,8))
        B = tf.shape(inputs)[0]
        x = tf.reshape(inputs, [B, 4, 4, 8])
        x = layers.GlobalAveragePooling2D()(x)
        print(f'Pre_Classify x shape is {x.shape}')
        outputs = layers.Dense(self.num_classes, name="head", kernel_initializer=initializers.Zeros(),activation="softmax")(x)  #(inputs[:, 0])
        print(f'Pre_Classify out shape is {outputs.shape}')
        model = Model(inputs, outputs)
        return model

    def Build_Succ_Classify(self):
        inputs = Input(shape=(16,8))
        B = tf.shape(inputs)[0]
        x = tf.reshape(inputs, [B, 4, 4, 8])
        x = layers.GlobalAveragePooling2D()(x)
        outputs = layers.Dense(self.num_classes, name="head", kernel_initializer=initializers.Zeros(),activation="softmax")(x)  #(inputs[:, 0])
        model = Model(inputs, outputs)
        return model


    def Mix(self, inputs):
        outputs = self.Succ_Embed(inputs)
        pre_outputs = self.Pre_blocks1(outputs)
        succ_outputs = self.Succ_blocks1(outputs)
        outputs = Choice1()([pre_outputs, succ_outputs])
        pre_outputs = self.Pre_blocks2(outputs)
        succ_outputs = self.Succ_blocks2(outputs)
        outputs = Choice1()([pre_outputs, succ_outputs])
        pre_outputs = self.Pre_blocks3(outputs)
        succ_outputs = self.Succ_blocks3(outputs)
        outputs = Choice1()([pre_outputs, succ_outputs])
        outputs = self.Succ_Classify(outputs)
        model = Model(inputs, outputs)
        return model

In [ ]:
random_seed = 666
random.seed(random_seed)  # set random seed for python
np.random.seed(random_seed)  # set random seed for numpy
tf.random.set_seed(random_seed)
print('gpu:',tf.test.is_gpu_available())

inputs = Input(shape=(8,8,1))
theseus = Theseus()
predecessor = theseus.Predecessor
predecessor.summary()
cosine_decay = tf.keras.optimizers.schedules.CosineDecay(
                initial_learning_rate=0.01, decay_steps=100)
exponential_decay = tf.keras.optimizers.schedules.ExponentialDecay(
                        initial_learning_rate=0.01, decay_steps=1000, decay_rate=0.96)

predecessor.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
   loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
   metrics=['sparse_categorical_accuracy'],
    loss_weights = None,
    sample_weight_mode = None,
    weighted_metrics = None,
    target_tensors = None
)
predecessor.summary()

feature_t = base_network(feature)
feature_t = tf.reshape(feature_t, [-1, 8, 8, 1])
print(f'before base network {feature_test.shape}')
feature_test_t = base_network(feature_test)
print(f"after base network {feature_test_t.shape}")
feature_test_t = tf.reshape(feature_test_t, [-1, 8, 8, 1])

predecessor.fit(feature_t, target, batch_size=1024, epochs=20, verbose=1,
          validation_split=0.25, validation_data=None, shuffle=True,
          class_weight=None, sample_weight=None, initial_epoch=0)
#predecessor.load_weights(r'.\weights\predecessor_TonIoT-NI_Siamese_gpu.h5') # 加载训练参数的结果

pre_test = predecessor.predict(feature_test_t, batch_size=1024)
print(pre_test.shape, target_test.shape)
print('ACC:',accuracy_score(target_test, np.argmax(pre_test, axis=1)))
print('Precision:',precision_score(target_test, np.argmax(pre_test, axis=1), average='weighted'))
print('Recall:',recall_score(target_test, np.argmax(pre_test, axis=1), average='weighted'))
print('F1_score:',f1_score(target_test, np.argmax(pre_test, axis=1), average='weighted'))